# LeNet-1989：从零手写卷积网络

这个 notebook 用纯 numpy 实现卷积、池化和反向传播，
然后在 MNIST 数据集的一个子集上训练一个简化版 LeNet。

**你会学到：**
- 卷积操作的内部实现
- 最大池化的实现
- 端到端的梯度流动
- 卷积核学到了什么（可视化）

In [ ]:
import numpy as np
import struct
import os
import urllib.request
import gzip
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for nbconvert
import matplotlib.pyplot as plt

np.random.seed(42)
print('numpy version:', np.__version__)

## 第一步：加载 MNIST 数据

MNIST 是一个包含 70000 张手写数字图片的数据集，每张 28×28 像素。
我们只用前 2000 张来加快训练速度。

In [ ]:
def load_mnist_images(path):
    with gzip.open(path, 'rb') as f:
        magic, n, rows, cols = struct.unpack('>IIII', f.read(16))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n, rows, cols).astype(np.float32) / 255.0

def load_mnist_labels(path):
    with gzip.open(path, 'rb') as f:
        magic, n = struct.unpack('>II', f.read(8))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.astype(np.int32)

def download_mnist(data_dir='data/mnist'):
    os.makedirs(data_dir, exist_ok=True)
    base = 'https://storage.googleapis.com/cvdf-datasets/mnist/'
    files = [
        'train-images-idx3-ubyte.gz',
        'train-labels-idx1-ubyte.gz',
        't10k-images-idx3-ubyte.gz',
        't10k-labels-idx1-ubyte.gz',
    ]
    for fname in files:
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f'Downloading {fname}...')
            urllib.request.urlretrieve(base + fname, fpath)
    return data_dir

data_dir = download_mnist()

X_train = load_mnist_images(os.path.join(data_dir, 'train-images-idx3-ubyte.gz'))
y_train = load_mnist_labels(os.path.join(data_dir, 'train-labels-idx1-ubyte.gz'))
X_test  = load_mnist_images(os.path.join(data_dir, 't10k-images-idx3-ubyte.gz'))
y_test  = load_mnist_labels(os.path.join(data_dir, 't10k-labels-idx1-ubyte.gz'))

# 只用前 2000 张训练，500 张测试（加快速度）
N_TRAIN, N_TEST = 2000, 500
X_train, y_train = X_train[:N_TRAIN], y_train[:N_TRAIN]
X_test,  y_test  = X_test[:N_TEST],   y_test[:N_TEST]

# 加一个 channel 维度：(N, 28, 28) → (N, 1, 28, 28)
X_train = X_train[:, np.newaxis, :, :]
X_test  = X_test[:,  np.newaxis, :, :]

print(f'训练集: {X_train.shape}, 标签: {y_train.shape}')
print(f'测试集: {X_test.shape},  标签: {y_test.shape}')

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i, 0], cmap='gray')
    ax.set_title(str(y_train[i]))
    ax.axis('off')
plt.suptitle('MNIST 样例图片')
plt.tight_layout()
plt.savefig('mnist_samples.png', dpi=72)
plt.close()
print('图片已保存到 mnist_samples.png')

## 第二步：手写卷积操作

卷积核（kernel）在图片上**滑动**，每次计算一个点积。

输入形状：`(batch, channels, height, width)`  
输出形状：`(batch, num_kernels, out_h, out_w)`

In [ ]:
def conv2d_forward(x, weights, bias):
    """
    x:       (N, C_in, H, W)
    weights: (C_out, C_in, kH, kW)
    bias:    (C_out,)
    returns: (N, C_out, H-kH+1, W-kW+1)  [valid 卷积，不补零]
    """
    N, C_in, H, W = x.shape
    C_out, _, kH, kW = weights.shape
    out_H = H - kH + 1
    out_W = W - kW + 1
    out = np.zeros((N, C_out, out_H, out_W), dtype=np.float32)
    for i in range(out_H):
        for j in range(out_W):
            # x_patch: (N, C_in, kH, kW)
            x_patch = x[:, :, i:i+kH, j:j+kW]
            # weights: (C_out, C_in, kH, kW) → 每个输出通道做一次点积
            # einsum: N,C_in,kH,kW × C_out,C_in,kH,kW → N,C_out
            out[:, :, i, j] = np.einsum('nckl,ockl->no', x_patch, weights) + bias
    return out

# 快速验证
x_test = np.ones((1, 1, 5, 5), dtype=np.float32)
w_test = np.ones((1, 1, 3, 3), dtype=np.float32)
b_test = np.zeros(1, dtype=np.float32)
out_test = conv2d_forward(x_test, w_test, b_test)
print('卷积输出形状:', out_test.shape)   # 期望 (1, 1, 3, 3)
print('卷积输出值（全1图×全1核）:', out_test[0, 0])  # 每个值都是 9.0（3×3=9个1相加）

In [ ]:
def conv2d_backward(x, weights, d_out):
    """
    计算卷积层的梯度。
    d_out:   上层传来的梯度 (N, C_out, out_H, out_W)
    返回:    (d_x, d_weights, d_bias)
    """
    N, C_in, H, W = x.shape
    C_out, _, kH, kW = weights.shape
    out_H, out_W = d_out.shape[2], d_out.shape[3]

    d_x = np.zeros_like(x)
    d_weights = np.zeros_like(weights)
    d_bias = d_out.sum(axis=(0, 2, 3))

    for i in range(out_H):
        for j in range(out_W):
            # d_out[:, :, i, j]: (N, C_out)
            g = d_out[:, :, i, j]  # (N, C_out)
            x_patch = x[:, :, i:i+kH, j:j+kW]  # (N, C_in, kH, kW)
            # 权重梯度: sum over N
            d_weights += np.einsum('no,nckl->ockl', g, x_patch)
            # 输入梯度
            d_x[:, :, i:i+kH, j:j+kW] += np.einsum('no,ockl->nckl', g, weights)

    return d_x, d_weights, d_bias

print('卷积反向传播函数定义完成')

## 第三步：手写最大池化

**最大池化**：在每个 2×2 小块里取最大值。  
反向传播时，梯度只流向那个产生最大值的位置，其他位置梯度为 0。

In [ ]:
def maxpool2d_forward(x, pool_size=2):
    """
    x: (N, C, H, W)
    returns: out (N, C, H//p, W//p), mask（记录最大值位置，用于反向传播）
    """
    N, C, H, W = x.shape
    p = pool_size
    out_H, out_W = H // p, W // p
    out = np.zeros((N, C, out_H, out_W), dtype=np.float32)
    mask = np.zeros_like(x, dtype=bool)
    for i in range(out_H):
        for j in range(out_W):
            patch = x[:, :, i*p:(i+1)*p, j*p:(j+1)*p]  # (N, C, p, p)
            max_val = patch.max(axis=(2, 3), keepdims=True)  # (N, C, 1, 1)
            out[:, :, i, j] = max_val[:, :, 0, 0]
            # 记录哪个位置是最大值
            mask[:, :, i*p:(i+1)*p, j*p:(j+1)*p] = (patch == max_val)
    return out, mask

def maxpool2d_backward(d_out, mask, pool_size=2):
    """
    梯度只流向曾经产生最大值的那个位置。
    """
    N, C, out_H, out_W = d_out.shape
    p = pool_size
    d_x = np.zeros((N, C, out_H*p, out_W*p), dtype=np.float32)
    for i in range(out_H):
        for j in range(out_W):
            g = d_out[:, :, i, j][:, :, np.newaxis, np.newaxis]  # (N, C, 1, 1)
            d_x[:, :, i*p:(i+1)*p, j*p:(j+1)*p] += g * mask[:, :, i*p:(i+1)*p, j*p:(j+1)*p]
    return d_x

# 验证池化
x_p = np.array([[[[1, 3, 2, 4],
                   [5, 6, 7, 8],
                   [9, 1, 2, 3],
                   [4, 5, 6, 7]]]], dtype=np.float32)  # (1,1,4,4)
out_p, _ = maxpool2d_forward(x_p)
print('池化输出:')
print(out_p[0, 0])  # 期望 [[6, 8], [9, 7]]

## 第四步：组装简化版 LeNet

我们的网络结构：
```
输入 (N, 1, 28, 28)
  → 卷积 C1: 4个 5×5 核 → (N, 4, 24, 24)
  → ReLU
  → 最大池化 2×2 → (N, 4, 12, 12)
  → 卷积 C2: 8个 5×5 核 → (N, 8, 8, 8)
  → ReLU
  → 最大池化 2×2 → (N, 8, 4, 4)
  → 展平 → (N, 128)
  → 全连接 → (N, 10)
  → Softmax
```

In [ ]:
def relu(x):
    return np.maximum(0, x)

def relu_backward(d_out, x):
    return d_out * (x > 0)

def softmax(x):
    x = x - x.max(axis=1, keepdims=True)  # 数值稳定
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy_loss(probs, y):
    N = y.shape[0]
    # 取每个样本正确类别的概率，加一个极小值防止 log(0)
    correct_probs = probs[np.arange(N), y]
    return -np.mean(np.log(correct_probs + 1e-9))

def cross_entropy_backward(probs, y):
    N = y.shape[0]
    d = probs.copy()
    d[np.arange(N), y] -= 1
    return d / N

print('激活函数和损失函数定义完成')

In [ ]:
class SimplifiedLeNet:
    def __init__(self):
        scale = 0.01
        # 卷积层1: 4个 5×5 的卷积核，输入通道1
        self.W1 = np.random.randn(4, 1, 5, 5).astype(np.float32) * scale
        self.b1 = np.zeros(4, dtype=np.float32)
        # 卷积层2: 8个 5×5 的卷积核，输入通道4
        self.W2 = np.random.randn(8, 4, 5, 5).astype(np.float32) * scale
        self.b2 = np.zeros(8, dtype=np.float32)
        # 全连接层: 128 → 10
        self.W3 = np.random.randn(128, 10).astype(np.float32) * scale
        self.b3 = np.zeros(10, dtype=np.float32)

    def forward(self, x):
        # 卷积块 1
        self.x = x
        self.c1 = conv2d_forward(x, self.W1, self.b1)   # (N,4,24,24)
        self.r1 = relu(self.c1)
        self.p1, self.mask1 = maxpool2d_forward(self.r1) # (N,4,12,12)
        # 卷积块 2
        self.c2 = conv2d_forward(self.p1, self.W2, self.b2)  # (N,8,8,8)
        self.r2 = relu(self.c2)
        self.p2, self.mask2 = maxpool2d_forward(self.r2)     # (N,8,4,4)
        # 展平 + 全连接
        N = x.shape[0]
        self.flat = self.p2.reshape(N, -1)    # (N, 128)
        self.logits = self.flat @ self.W3 + self.b3  # (N, 10)
        self.probs = softmax(self.logits)
        return self.probs

    def backward(self, y, lr=0.01):
        # 输出层梯度
        d = cross_entropy_backward(self.probs, y)  # (N,10)
        # 全连接层
        dW3 = self.flat.T @ d
        db3 = d.sum(axis=0)
        d_flat = d @ self.W3.T  # (N, 128)
        # 展平 → 池化2
        d_p2 = d_flat.reshape(self.p2.shape)
        # 池化2 反向
        d_r2 = maxpool2d_backward(d_p2, self.mask2)
        # ReLU2 反向
        d_c2 = relu_backward(d_r2, self.c2)
        # 卷积2 反向
        d_p1, dW2, db2 = conv2d_backward(self.p1, self.W2, d_c2)
        # 池化1 反向
        d_r1 = maxpool2d_backward(d_p1, self.mask1)
        # ReLU1 反向
        d_c1 = relu_backward(d_r1, self.c1)
        # 卷积1 反向
        _, dW1, db1 = conv2d_backward(self.x, self.W1, d_c1)
        # 梯度下降更新
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W3 -= lr * dW3
        self.b3 -= lr * db3

print('SimplifiedLeNet 定义完成')

## 第五步：训练

我们用**小批量梯度下降**（Mini-batch SGD）训练 10 个 epoch。

每个 epoch 里，数据被随机打乱，每次取 32 张图（一个 batch）更新一次权重。

In [ ]:
model = SimplifiedLeNet()
EPOCHS = 10
BATCH_SIZE = 32
LR = 0.05

train_losses = []
train_accs = []

for epoch in range(EPOCHS):
    # 打乱数据
    idx = np.random.permutation(N_TRAIN)
    X_shuf, y_shuf = X_train[idx], y_train[idx]

    epoch_loss = 0.0
    epoch_correct = 0
    n_batches = 0

    for start in range(0, N_TRAIN, BATCH_SIZE):
        xb = X_shuf[start:start+BATCH_SIZE]
        yb = y_shuf[start:start+BATCH_SIZE]
        probs = model.forward(xb)
        loss = cross_entropy_loss(probs, yb)
        model.backward(yb, lr=LR)
        preds = probs.argmax(axis=1)
        epoch_loss += loss
        epoch_correct += (preds == yb).sum()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    acc = epoch_correct / N_TRAIN * 100
    train_losses.append(avg_loss)
    train_accs.append(acc)
    print(f'Epoch {epoch+1:2d}/{EPOCHS}  loss={avg_loss:.4f}  train_acc={acc:.1f}%')

print('\n训练完成！')

## 第六步：评估测试集准确率

In [ ]:
# 在测试集上评估（分批，避免内存溢出）
test_correct = 0
for start in range(0, N_TEST, BATCH_SIZE):
    xb = X_test[start:start+BATCH_SIZE]
    yb = y_test[start:start+BATCH_SIZE]
    probs = model.forward(xb)
    preds = probs.argmax(axis=1)
    test_correct += (preds == yb).sum()

test_acc = test_correct / N_TEST * 100
print(f'测试集准确率: {test_acc:.1f}%')

# 验证：至少比随机（10%）好
assert test_acc > 20, f'准确率太低（{test_acc:.1f}%），请检查实现'

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(range(1, EPOCHS+1), train_losses, 'b-o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('训练损失')
ax2.plot(range(1, EPOCHS+1), train_accs, 'r-o')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('准确率 (%)')
ax2.set_title('训练准确率')
ax2.axhline(y=test_acc, color='green', linestyle='--', label=f'测试准确率 {test_acc:.1f}%')
ax2.legend()
plt.tight_layout()
plt.savefig('training_curve.png', dpi=72)
plt.close()
print('训练曲线已保存到 training_curve.png')

## 第七步：可视化卷积核

卷积核是**网络学到的特征检测器**。
第一层的 4 个 5×5 卷积核，让我们看看它们学到了什么形状。

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(8, 2))
for i, ax in enumerate(axes):
    kernel = model.W1[i, 0]  # (5, 5)
    vmax = np.abs(kernel).max()
    ax.imshow(kernel, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(f'核 {i+1}')
    ax.axis('off')
plt.suptitle('第一卷积层学到的 4 个卷积核')
plt.tight_layout()
plt.savefig('kernels.png', dpi=72)
plt.close()
print('卷积核可视化已保存到 kernels.png')
print('（蓝色=负权重，红色=正权重）')

## 第八步：LeNet 的意义

我们刚才手动实现了：
- **卷积**：用小卷积核扫描整张图，每个位置共享同一组权重
- **池化**：缩小特征图，保留最重要的信息
- **端到端反向传播**：梯度从输出一路流回卷积核

1989 年，LeCun 的团队第一次证明这套机制能够在真实的手写数字识别上工作。

**下一步**：加载 pytorch 或 tensorflow，把我们的手写版和库函数版对比：

In [ ]:
# 总结本 notebook 的关键数字
print('=== LeNet-1989 实验总结 ===')
print(f'  训练样本数: {N_TRAIN}')
print(f'  测试样本数: {N_TEST}')
print(f'  卷积层1 参数量: 4×1×5×5 + 4 = {4*1*5*5+4}')
print(f'  卷积层2 参数量: 8×4×5×5 + 8 = {8*4*5*5+8}')
print(f'  全连接层参数量: 128×10 + 10 = {128*10+10}')
total_params = 4*1*5*5+4 + 8*4*5*5+8 + 128*10+10
print(f'  总参数量: {total_params}（比全连接网络少得多！）')
print(f'  最终测试准确率: {test_acc:.1f}%')
print(f'  随机猜测基准: 10.0%')
print(f'  提升: +{test_acc-10:.1f}%')